# 06B1 CHARMM/GROMACS vacuum check

CHARMM/GROMACS: polymer CGenFF (from `05B_charmm_cgenff_parameters.ipynb`) · protein CHARMM36m · CHARMM TIP3P + SOD/CLA · engine GROMACS.

This notebook checks the CGenFF polymer **on its own, without solvent**, using the native `gromacs/` files in a CHARMM-GUI Ligand Reader & Modeler download (`topol.top`, `charmm36.itp`, `LIG.itp`) and its coordinates (`ligandrm.pdb`).

Dry runs are sanity checks: the topology loads, the charges and atom types are what CGenFF assigned, the stereocentres are still R, and the structure minimises. They are not physical results. For the solvated system (polymer in water, or enzyme + polymer) use `06B2_charmm_gromacs_solvated_system.ipynb`.

GROMACS' Verlet scheme has no true vacuum mode, so the polymer sits alone in a periodic box larger than twice the 1.2 nm cut-off.

## 1. Inputs

- `LIGAND_READER_DIR`: the unpacked Ligand Reader & Modeler download (the folder that contains `gromacs/`, `lig/` and `ligandrm.pdb`). It is only read.
- `SDF_PATH`: the SDF you uploaded to CHARMM-GUI. It supplies bond orders for the stereocentre check.
- `DRY_DIR`: a **new** folder for the dry system. It must not exist yet and must not be inside the download.

In [ ]:
from pathlib import Path
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))
from iphasimulator.charmmgui_import import (
    format_checks,
    prepare_dry_ligand_folder,
    run_dry_minimization,
    validate_dry_ligand_folder,
)

LIGAND_READER_DIR = None  # e.g. Path("/path/to/PHO4_ligand_generator_charmm-gui-XXXXXXXXXX")
SDF_PATH = None           # e.g. repo_root / "examples/output/polymer_structures/PHO4_R.sdf"
DRY_DIR = None            # e.g. repo_root / "examples/output/charmm_gromacs/PHO4_dry"

## 2. Prepare the dry folder

Copies the three topology files, writes `lig.gro` (molecule centred in a cubic box of its size + 2 × 1.7 nm) and `dry_minimization.mdp` (steepest descent, CHARMM cut-offs).

In [ ]:
if LIGAND_READER_DIR is None or DRY_DIR is None:
    print("Set LIGAND_READER_DIR and DRY_DIR in step 1 to prepare the dry folder.")
else:
    dry_dir = prepare_dry_ligand_folder(LIGAND_READER_DIR, DRY_DIR)
    print("Prepared:", dry_dir)
    for path in sorted(dry_dir.iterdir()):
        print("  ", path.name)

## 3. Read-only checks

Each line is PASS, FAIL or SKIP with the evidence behind it: CHARMM-style `[ defaults ]`, include files, atom counts, net charge, CGenFF-style atom types, atom order, and the CIP label of every stereocentre (from the GROMACS coordinates, with bond orders taken from the SDF).

In [ ]:
if DRY_DIR is None or not Path(DRY_DIR).is_dir():
    print("Prepare the dry folder in step 2 first.")
else:
    print(format_checks(validate_dry_ligand_folder(DRY_DIR, sdf_path=SDF_PATH)))

## 4. Optional: dry minimisation

Runs `gmx grompp` and `gmx mdrun` in `DRY_DIR` only (seconds for a single oligomer). A converged minimisation reports a finite potential energy and a maximum force below the `emtol` of 100 kJ mol⁻¹ nm⁻¹.

In [ ]:
RUN_DRY_MINIMIZATION = False

if not RUN_DRY_MINIMIZATION:
    print("Set RUN_DRY_MINIMIZATION = True to minimise the dry polymer with GROMACS.")
else:
    result = run_dry_minimization(DRY_DIR)
    print(f"Potential energy: {result.potential_energy} kJ/mol")
    print(f"Maximum force:    {result.maximum_force} kJ/mol/nm")
    print("Log:", result.log_path)

## Next step

Build the solvated system in CHARMM-GUI Solution Builder (see 05B) and prepare and validate it in `06B2_charmm_gromacs_solvated_system.ipynb`.